In [1]:
import pandas as pd
import numpy as np 
import os

In [2]:
df_matches = pd.read_csv("/kaggle/input/datasets/patrickb1912/ipl-complete-dataset-20082020/matches.csv")
df_deliveries = pd.read_csv("/kaggle/input/datasets/patrickb1912/ipl-complete-dataset-20082020/deliveries.csv")

In [3]:
print(df_matches.shape)
print(df_deliveries.shape)

(1095, 20)
(260920, 17)


In [4]:
print(df_matches.columns)
print(df_deliveries.columns)

Index(['id', 'season', 'city', 'date', 'match_type', 'player_of_match',
       'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner',
       'result', 'result_margin', 'target_runs', 'target_overs', 'super_over',
       'method', 'umpire1', 'umpire2'],
      dtype='object')
Index(['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball',
       'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs',
       'total_runs', 'extras_type', 'is_wicket', 'player_dismissed',
       'dismissal_kind', 'fielder'],
      dtype='object')


# Step 2 : standardize columns names

In [5]:
matches_rename = {
    'id' : 'match_id',
    'result_margin' : 'win_margin'
}

deliveries_rename = {
    'match_id' : 'match_id',
    'batter' : 'batsman'
}

df_matches = df_matches.rename(columns={k: v for k, v in matches_rename.items()
                                       if k in df_matches.columns})
df_deliveries = df_deliveries.rename(columns={k: v for k, v in deliveries_rename.items()
                                             if k in df_deliveries.columns})

print("matches.csv columns :",df_matches.columns.tolist())
print("deliveries.csv columns :",df_deliveries.columns.tolist())

matches.csv columns : ['match_id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'win_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']
deliveries.csv columns : ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batsman', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']


# Step3: cleaning matches.csv

In [6]:
# fixing date column 
df_matches['date'] = pd.to_datetime(df_matches['date'],errors='coerce')
df_matches['year'] = df_matches['date'].dt.year    #  .dt.year --- extracting year from date making new column

# fixing season column
df_matches['season'] = df_matches['season'].astype(str).str[:4].astype(int,errors='ignore')
df_matches['season'] = pd.to_numeric(df_matches['season'],errors='coerce')


In [7]:
# normalizing teams names..

team_rename_map = {
    'Delhi Daredevils'          : 'Delhi Capitals',
    'Deccan Chargers'           : 'Sunrisers Hyderabad',
    'Rising Pune Supergiant'    : 'Rising Pune Supergiants',
    'Kings XI Punjab'           : 'Punjab Kings',
    'Pune Warriors'             : 'Pune Warriors India',
    'Royal Challengers Bangalore' : 'Royal Challengers Bengaluru',
}
 
for col in ['team1', 'team2', 'toss_winner', 'winner']:
    if col in df_matches.columns:
        df_matches[col] = df_matches[col].replace(team_rename_map)
 
print(f"Matches after date fix: {len(df_matches)} rows")
print(f"Season range: {df_matches['season'].min()} – {df_matches['season'].max()}")
print(f"Teams: {sorted(df_matches['team1'].dropna().unique())}")

Matches after date fix: 1095 rows
Season range: 2007 – 2024
Teams: ['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors India', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiants', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']


In [8]:
# creating toss_won_match column...
df_matches['toss_won_match'] = (df_matches['toss_winner'] == df_matches['winner']).astype(int)

In [9]:
# dropping rows where winner is null
before = len(df_matches)
df_matches = df_matches.dropna(subset=['winner'])
print(f"Dropped {before - len(df_matches)} abandoned/no-result matches")
print(f"Remaining matches: {len(df_matches)}")

Dropped 5 abandoned/no-result matches
Remaining matches: 1090


# Step 4: cleaning deliveries.csv 

In [10]:
# Normalize team names in deliveries too ---
for col in ['batting_team', 'bowling_team']:
    if col in df_deliveries.columns:
        df_deliveries[col] = df_deliveries[col].replace(team_rename_map)


# Ensure numeric columns are correct ---
for col in ['batsman_runs', 'extra_runs', 'total_runs', 'wide_runs',
            'bye_runs', 'legbye_runs', 'noball_runs']:
    if col in df_deliveries.columns:
        df_deliveries[col] = pd.to_numeric(df_deliveries[col], errors='coerce').fillna(0)
 
print(f"Deliveries after cleaning: {len(df_deliveries)} rows")
print(f"Unique match IDs in deliveries: {df_deliveries['match_id'].nunique()}")
 

Deliveries after cleaning: 260920 rows
Unique match IDs in deliveries: 1095


# Step 5: Building match-level aggregrates from deliveries.

In [11]:
# Per match, per inning aggregations
match_innings_agg = df_deliveries.groupby(['match_id', 'inning']).agg(
    total_runs_scored = ('total_runs',     'sum'),
    total_wickets     = ('is_wicket',      'sum'),
    total_fours       = ('batsman_runs',   lambda x: (x == 4).sum()),
    total_sixes       = ('batsman_runs',   lambda x: (x == 6).sum()),
    total_extras      = ('extra_runs',     'sum'),
    balls_bowled      = ('ball',           'count'),
).reset_index()


# Pivot: inning 1 and inning 2 become separate columns
inn1 = match_innings_agg[match_innings_agg['inning'] == 1].copy()
inn2 = match_innings_agg[match_innings_agg['inning'] == 2].copy()
 
inn1 = inn1.rename(columns={
    'total_runs_scored' : 'inn1_runs',
    'total_wickets'     : 'inn1_wickets',
    'total_fours'       : 'inn1_fours',
    'total_sixes'       : 'inn1_sixes',
    'total_extras'      : 'inn1_extras',
    'balls_bowled'      : 'inn1_balls',
}).drop(columns=['inning'])
 
inn2 = inn2.rename(columns={
    'total_runs_scored' : 'inn2_runs',
    'total_wickets'     : 'inn2_wickets',
    'total_fours'       : 'inn2_fours',
    'total_sixes'       : 'inn2_sixes',
    'total_extras'      : 'inn2_extras',
    'balls_bowled'      : 'inn2_balls',
}).drop(columns=['inning'])
 
# Merge inning stats back into one row per match
match_stats = inn1.merge(inn2, on='match_id', how='outer')
match_stats['total_match_runs'] = match_stats['inn1_runs'].fillna(0) + match_stats['inn2_runs'].fillna(0)
 
print(f"Match-level aggregates built: {len(match_stats)} rows")
print(match_stats.head(3))
 

Match-level aggregates built: 1095 rows
   match_id  inn1_runs  inn1_wickets  inn1_fours  inn1_sixes  inn1_extras  \
0    335982        222             3          15          14           17   
1    335983        240             5          20          16            6   
2    335984        129             8          14           3            7   

   inn1_balls  inn2_runs  inn2_wickets  inn2_fours  inn2_sixes  inn2_extras  \
0         124       82.0          10.0         3.0         3.0         19.0   
1         124      207.0           4.0        18.0         9.0         11.0   
2         122      132.0           1.0        18.0         1.0         10.0   

   inn2_balls  total_match_runs  
0       101.0             304.0  
1       124.0             447.0  
2        97.0             261.0  


# Step 6: Merge matches + match stats

In [12]:
 
df = df_matches.merge(match_stats, on='match_id', how='left')
 
print(f"After merge: {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
 
# Guard: check merge quality
unmatched = df['inn1_runs'].isna().sum()
if unmatched > 10:
    print(f"\n⚠️  WARNING: {unmatched} matches have no delivery data.")
    print("   Check that match IDs match between both files.")
 



After merge: 1090 rows
Columns: ['match_id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'win_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2', 'year', 'toss_won_match', 'inn1_runs', 'inn1_wickets', 'inn1_fours', 'inn1_sixes', 'inn1_extras', 'inn1_balls', 'inn2_runs', 'inn2_wickets', 'inn2_fours', 'inn2_sixes', 'inn2_extras', 'inn2_balls', 'total_match_runs']


# Step 7: Creating calculated columns

In [13]:
# Adding : did chasing team win ?
df['chased_successfully'] = np.where(
    (df['toss_decision'] == 'field') & (df['toss_won_match'] == 1),1,
    np.where((df['toss_decision'] == 'bat') & (df['toss_won_match'] == 0),1,0)
)


# Win margin category 
def classify_win_margin(row):
    if row['result'] == 'runs':
        runs = row['win_margin']
        if runs >= 100: return 'Dominant (100+ runs)'
        elif runs >= 50: return 'Comfortable (50-99 runs)'
        else:            return 'Close (<50 runs)'
    elif row['result'] == 'wickets':
        wkts = row['win_margin']
        if wkts >= 8:  return 'Dominant (8+ wkts)'
        elif wkts >= 5: return 'Comfortable (5-7 wkts)'
        else:           return 'Close (1-4 wkts)'
    return 'Super Over / No Result'

df['win_margin_category'] = df.apply(classify_win_margin, axis=1)

# Season Era.....
def classify_era(season):
    if season<=2013: return 'Early IPL (2007-13)'
    elif season <=2019: return 'Growth Era (2014-2019)'
    else: return 'Modern IPL(2020 +)'

df['era'] = df['season'].apply(classify_era)

# High Scoring match flag( matlab run > 350)
df['high_scoring_match'] = (df['total_match_runs']>350).astype(int)

 
print("Created columns: toss_won_match, chased_successfully, win_margin_category, era, high_scoring_match")
 

Created columns: toss_won_match, chased_successfully, win_margin_category, era, high_scoring_match


# Step 8: Handling Missing Values

In [14]:
print(df.isnull().sum()[df.isnull().sum()>0])

# solving city parts
df['city'] = df['city'].fillna('Unknown')

# player_of_match - rare nulls, fill Unknown
df['player_of_match'] = df['player_of_match'].fillna('Unknown')

# Umpires - not critical, fill Unknown
for col in ['umpire1','umpire2','umpire3']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')


city            51
win_margin      14
method        1069
dtype: int64


# step 9: Final checking & saving


In [15]:
print("\n" + "=" * 65)
print("FINAL DATASET SUMMARY")
print("=" * 65)
print(f"Total matches        : {len(df)}")
print(f"Season range         : {df['season'].min()} – {df['season'].max()}")
print(f"Unique teams         : {df['team1'].nunique()}")
print(f"Unique venues        : {df['venue'].nunique()}")
print(f"Avg inn1 runs        : {df['inn1_runs'].mean():.1f}")
print(f"Avg inn2 runs        : {df['inn2_runs'].mean():.1f}")
print()
print("Toss decision split:")
print(df['toss_decision'].value_counts())
print()
print("Win margin categories:")
print(df['win_margin_category'].value_counts())
print()
print("Era split:")
print(df['era'].value_counts())


FINAL DATASET SUMMARY
Total matches        : 1090
Season range         : 2007 – 2024
Unique teams         : 14
Unique venues        : 58
Avg inn1 runs        : 165.7
Avg inn2 runs        : 152.4

Toss decision split:
toss_decision
field    700
bat      390
Name: count, dtype: int64

Win margin categories:
win_margin_category
Close (<50 runs)            405
Comfortable (5-7 wkts)      332
Dominant (8+ wkts)          142
Close (1-4 wkts)            104
Comfortable (50-99 runs)     82
Super Over / No Result       14
Dominant (100+ runs)         11
Name: count, dtype: int64

Era split:
era
Early IPL (2007-13)       397
Growth Era (2014-2019)    355
Modern IPL(2020 +)        338
Name: count, dtype: int64


In [16]:
# Saving match-leve cleaned file
df.to_csv("ipl_cleaned.csv",index=False)
print("\n Saved: ipl_cleaned.csv")

# Also saving cleaned deliveries
df_deliveries['season'] = df_deliveries['match_id'].map(
    df_matches.set_index('match_id')['season']
)

df_deliveries.to_csv("ipl_deliveries_cleande.csv",index=False)
print("\n Saved: ipl deliveries cleaned csv")


 Saved: ipl_cleaned.csv

 Saved: ipl deliveries cleaned csv
